# Topic 3: Error Handling & Pipeline Monitoring

> **Phase 5 → Week 9 → Topic 3**

---

## Why Error Handling in ETL is Hard

```
A Spark job can fail at many levels:

  Level 1: Infrastructure   → node crash, network split, disk full
           (Spark retries tasks automatically)

  Level 2: Data parsing     → malformed records, type errors
           (handle with PERMISSIVE + dead letter, Week 8)

  Level 3: Business logic   → divide by zero, lookup miss, bad join result
           (handle with try/except in UDFs, validation rules)

  Level 4: Pipeline logic   → wrong watermark, duplicate write, schema mismatch
           (handle with idempotent design + schema checks)

  Level 5: Resource         → OOM, executor lost, GC pause timeout
           (handle with partition tuning, memory config, retries)
```

---

## Interview Cheat Sheet

**Q: How do you handle exceptions inside a Spark UDF?**
> Wrap the UDF body in try/except and return `None` (null) for failed rows. Separately, use an accumulator to count failures or return an error code column. Never let an exception propagate out of a UDF — it kills the executor task. Log the error type/message to a broadcast-accessible logger or accumulator for debugging.

**Q: What pipeline metrics should you track in production?**
> At minimum: (1) row counts at each stage (Bronze, Silver, Gold), (2) rejection rate (bad records / total), (3) duration per stage, (4) shuffle bytes and spill, (5) data freshness (lag between source event time and load time). Alert on: rejection rate > threshold, row count drops > X%, stage duration > Y minutes, any executor failures.

**Q: What is a checkpoint in a Spark Streaming or long-running ETL context?**
> A checkpoint saves the query state (offsets, watermarks, in-progress aggregations) to a durable location so a failed job can resume from where it stopped rather than from scratch. In Structured Streaming: `.writeStream.option('checkpointLocation', '/path')`. In batch ETL: saving the watermark + output atomically to a DB serves the same purpose.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, logging

spark = SparkSession.builder \
    .appName("Week9 - Error Handling") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
spark.sparkContext.setLogLevel("WARN")

# Python logger for the driver
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("ETL")
print("Ready")

## Part 1: Error Handling in UDFs

In [ ]:
from pyspark.sql.functions import udf

# BAD: unhandled exception kills the executor task
@udf(DoubleType())
def bad_divide(a, b):
    return a / b  # ZeroDivisionError if b=0 → task fails!

# GOOD: always handle exceptions in UDFs, return null for errors
@udf(DoubleType())
def safe_divide(a, b):
    try:
        if b is None or b == 0:
            return None
        return float(a) / float(b)
    except Exception:
        return None

df = spark.createDataFrame([
    ("O001", 100.0, 4.0),
    ("O002", 200.0, 0.0),   # division by zero
    ("O003", 150.0, None),  # null divisor
    ("O004", 300.0, 3.0),
], ["order_id", "revenue", "units"])

result = df.withColumn("price_per_unit", safe_divide(F.col("revenue"), F.col("units")))
print("Safe UDF (errors → null, no crash):")
result.show()

# Track UDF failures with accumulator
udf_errors = sc.accumulator(0)

@udf(DoubleType())
def safe_divide_counted(a, b):
    try:
        if b is None or b == 0:
            udf_errors.add(1)
            return None
        return float(a) / float(b)
    except Exception:
        udf_errors.add(1)
        return None

df.withColumn("price", safe_divide_counted(F.col("revenue"), F.col("units"))).count()
print(f"UDF error count: {udf_errors.value}")

## Part 2: Pipeline Metrics Collection

In [ ]:
import json
from dataclasses import dataclass, field, asdict
from typing import Dict, List
from datetime import datetime

@dataclass
class StageMetrics:
    name: str
    start_time: str = ""
    end_time: str = ""
    duration_s: float = 0.0
    input_rows: int = 0
    output_rows: int = 0
    rejected_rows: int = 0
    errors: List[str] = field(default_factory=list)

    @property
    def rejection_rate(self):
        return (self.rejected_rows / self.input_rows * 100) if self.input_rows > 0 else 0

@dataclass
class PipelineRun:
    job_name: str
    run_date: str
    stages: Dict[str, StageMetrics] = field(default_factory=dict)
    status: str = "running"  # running / success / failed

    def start_stage(self, name: str) -> StageMetrics:
        m = StageMetrics(name=name, start_time=datetime.now().isoformat())
        self.stages[name] = m
        return m

    def end_stage(self, m: StageMetrics):
        m.end_time = datetime.now().isoformat()
        from datetime import datetime as dt
        m.duration_s = round(
            (dt.fromisoformat(m.end_time) - dt.fromisoformat(m.start_time)).total_seconds(), 2
        )

    def summary(self):
        print(f"\nPipeline: {self.job_name} | Date: {self.run_date} | Status: {self.status}")
        print("-" * 70)
        for name, m in self.stages.items():
            print(f"  {name:25s} {m.input_rows:>8,} in → {m.output_rows:>8,} out  "
                  f"({m.rejection_rate:.1f}% rejected)  {m.duration_s:.2f}s")
            for err in m.errors:
                print(f"    ⚠ {err}")

print("PipelineRun metrics class defined")

In [ ]:
# Use the metrics class in a real ETL pipeline
orders_raw = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
products   = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)

run = PipelineRun(job_name="orders_daily_etl", run_date="2024-01-20")

# Stage 1: ingest
m1 = run.start_stage("01_ingest")
m1.input_rows = orders_raw.count()
m1.output_rows = m1.input_rows
run.end_stage(m1)

# Stage 2: validate
m2 = run.start_stage("02_validate")
m2.input_rows = m1.output_rows

valid_orders = orders_raw.filter(
    F.col("amount").isNotNull() &
    (F.col("amount") > 0) &
    F.col("customer_id").isNotNull()
)
m2.output_rows = valid_orders.count()
m2.rejected_rows = m2.input_rows - m2.output_rows
if m2.rejection_rate > 10:
    m2.errors.append(f"HIGH rejection rate: {m2.rejection_rate:.1f}%")
run.end_stage(m2)

# Stage 3: enrich
m3 = run.start_stage("03_enrich")
m3.input_rows = m2.output_rows
enriched = valid_orders.join(F.broadcast(products.select("product_id", "category")),
                              on="product_id", how="left")
m3.output_rows = enriched.count()
run.end_stage(m3)

run.status = "success"
run.summary()

## Part 3: Alerting Thresholds

In [ ]:
# Production alerting rules
def check_pipeline_health(run: PipelineRun, prev_row_count: int = None) -> bool:
    """Check pipeline metrics against thresholds. Returns True if healthy."""
    alerts = []

    for name, m in run.stages.items():
        # Alert: rejection rate > 5%
        if m.rejection_rate > 5:
            alerts.append(f"[{name}] REJECTION RATE {m.rejection_rate:.1f}% > 5% threshold")

        # Alert: stage took > 30 minutes
        if m.duration_s > 1800:
            alerts.append(f"[{name}] SLOW STAGE {m.duration_s/60:.1f} min > 30 min threshold")

        # Alert: zero rows out
        if m.output_rows == 0 and m.input_rows > 0:
            alerts.append(f"[{name}] ZERO OUTPUT (all rows rejected or failed)")

    # Alert: row count dropped > 20% vs previous run
    if prev_row_count is not None:
        last_stage = list(run.stages.values())[-1]
        drop_pct = (prev_row_count - last_stage.output_rows) / prev_row_count * 100
        if drop_pct > 20:
            alerts.append(f"ROW COUNT DROPPED {drop_pct:.1f}% vs previous run")

    if alerts:
        print("🚨 PIPELINE ALERTS:")
        for a in alerts:
            print(f"  ✗ {a}")
        return False
    else:
        print("✓ Pipeline health: all checks passed")
        return True

# Check our run
check_pipeline_health(run, prev_row_count=100)  # simulate previous run had 100 rows

## Part 4: Retry Logic and Fault Tolerance

In [ ]:
import time

def run_with_retry(fn, max_retries=3, backoff_s=2, job_name="job"):
    """Retry a function with exponential backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            logger.info(f"[{job_name}] Attempt {attempt}/{max_retries}")
            result = fn()
            logger.info(f"[{job_name}] Success on attempt {attempt}")
            return result
        except Exception as e:
            if attempt == max_retries:
                logger.error(f"[{job_name}] Failed after {max_retries} attempts: {e}")
                raise
            wait = backoff_s * (2 ** (attempt - 1))  # exponential: 2s, 4s, 8s
            logger.warning(f"[{job_name}] Attempt {attempt} failed: {e}. Retrying in {wait}s")
            time.sleep(wait)

# Demo: a function that fails the first 2 times
attempt_counter = {"n": 0}

def flaky_etl():
    attempt_counter["n"] += 1
    if attempt_counter["n"] < 3:
        raise ConnectionError("Source DB connection timed out")
    return "ETL completed successfully"

result = run_with_retry(flaky_etl, max_retries=3, backoff_s=0, job_name="orders_etl")
print(f"Result: {result}")

In [ ]:
print("""
Spark's Built-in Fault Tolerance:
════════════════════════════════════════════════════════════════
Spark automatically retries at multiple levels:

Task level:
  spark.task.maxFailures = 4 (default)
  A task fails → Spark retries on another executor
  After maxFailures attempts → stage fails

Stage level:
  spark.stage.maxConsecutiveAttempts = 4 (default)
  A stage fails → Spark retries the whole stage (re-reads input)

Application level (external orchestrator):
  Airflow: retries=3, retry_delay=timedelta(minutes=5)
  Your pipeline code: run_with_retry() wraps the whole job

Important: retries are only safe if your writes are IDEMPOTENT.
If a task retries after partial success → without idempotency you get duplicates.
Solution: always use overwrite/replaceWhere/MERGE, never plain append.

Speculative execution:
  spark.speculation = true (disabled by default)
  Spark launches duplicate tasks for slow tasks — uses whichever finishes first.
  Helps with stragglers but can cause double-processing for non-idempotent writes.
  Use with caution.
════════════════════════════════════════════════════════════════
""")

## Part 5: Logging Best Practices

In [ ]:
# Structured logging for ETL pipelines
import json as json_lib
from datetime import datetime

def log_event(level: str, job: str, stage: str, message: str, **kwargs):
    """Emit a structured JSON log event (easy to parse with log aggregators)."""
    event = {
        "timestamp": datetime.now().isoformat(),
        "level": level,
        "job": job,
        "stage": stage,
        "message": message,
        **kwargs
    }
    print(json_lib.dumps(event))

# Usage in a pipeline
log_event("INFO",  "orders_etl", "ingest",   "Read source",    rows=20)
log_event("INFO",  "orders_etl", "validate", "Validation done", valid=18, rejected=2, rejection_pct=10.0)
log_event("WARN",  "orders_etl", "validate", "High rejection",  rejection_pct=10.0, threshold=5.0)
log_event("INFO",  "orders_etl", "write",    "Write complete",  rows=18, path="/output/orders", duration_s=1.2)

print("""
Structured (JSON) logging advantages:
  ✓ Machine-parseable — log aggregators (ELK, Datadog, CloudWatch) can index fields
  ✓ Queryable — 'find all events where rejection_pct > 5'
  ✓ Alertable — trigger alerts on specific field values
  ✓ Consistent — same format across all ETL jobs

What to log in every ETL job:
  ✓ Job start: job_name, run_date, input parameters
  ✓ Per stage: row counts in/out, duration, rejection rate
  ✓ Any data quality alerts
  ✓ Job end: status (success/failed), total duration, output location
  ✓ On failure: exception type, message, stage name, partial counts
""")

## Exercises

1. Write a UDF that parses a date string in multiple formats (`yyyy-MM-dd`, `MM/dd/yyyy`, `dd-MM-yyyy`). For unrecognized formats, return null and count failures with an accumulator.
2. Add a `check_row_count_drop` alert to `check_pipeline_health` that fires if today's output rows are < 80% of yesterday's.
3. What is speculative execution (`spark.speculation`)? When would you enable it and why can it cause problems with non-idempotent writes?
4. A pipeline fails at stage 3 of 5. With idempotent design, can you safely re-run from stage 1? What about re-running just stages 3-5?
5. Write a `StageTimer` context manager that automatically records start/end time and duration for a pipeline stage.

In [ ]:
# Exercise 5: StageTimer context manager
from contextlib import contextmanager
import time

@contextmanager
def stage_timer(name: str):
    """Context manager that logs stage timing automatically."""
    t0 = time.time()
    log_event("INFO", "etl", name, "Stage started")
    try:
        yield
        duration = round(time.time() - t0, 2)
        log_event("INFO", "etl", name, "Stage completed", duration_s=duration)
    except Exception as e:
        duration = round(time.time() - t0, 2)
        log_event("ERROR", "etl", name, "Stage failed", duration_s=duration, error=str(e))
        raise

# Usage
with stage_timer("validate"):
    orders_raw = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
    valid = orders_raw.filter(F.col("amount") > 0)
    count = valid.count()

with stage_timer("enrich"):
    products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)
    enriched = valid.join(F.broadcast(products.select("product_id", "category")), on="product_id")
    enriched.count()